# 3. Training and tuning LightGBM models

## Goal

Train one cross-validated classifier per phase and one regressor per
transition temperature, inspect fold-wise importance and learning curves,
and tune the original LightGBM search space with Optuna.

## Setup

Full fitting and tuning are controlled by independent run switches. Both
are disabled by default because the five-molecule example table is intended
for workflow verification rather than statistical estimation.

### 1. Configure the modelling table

**Caution.** LightGBM accepts numeric missing descriptor values, but object-valued error tokens must be converted during descriptor preparation.

**Extension.** Ablation studies use `Shape`, `Electric`, `Flexible` and the combined or subtracted families from the package catalogue.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
EXAMPLE_DATA = PROJECT_ROOT / "examples" / "data"
RESULT_DIRECTORY = PROJECT_ROOT / "results"
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

from lc_qspr import PHASE_TARGETS, TEMPERATURE_TARGETS
from lc_qspr.descriptors import descriptor_families, select_descriptor_names
from lc_qspr.preprocessing import read_table, separate_chiral_phases

DATA_FILE = EXAMPLE_DATA / "sample_desc.csv"
RUN_LIGHTGBM_TRAINING = False
RUN_OPTUNA_TUNING = False
DESCRIPTOR_FAMILY = "All"
N_SPLITS = 5

table, _ = separate_chiral_phases(read_table(DATA_FILE))
families = descriptor_families()
feature_names = select_descriptor_names(table, DESCRIPTOR_FAMILY)
phase_targets = [target for target in PHASE_TARGETS if target in table]
temperature_targets = [target for target in TEMPERATURE_TARGETS if target in table]
print(table.shape, len(feature_names))

(5, 1654) 1613


### 2. Create shared folds

**Caution.** Using different folds for phase and temperature models complicates a paired comparison; this example shares the multi-label phase split.

**Extension.** Persist the integer indices if later runs will be performed on another machine.

In [2]:
from lc_qspr.evaluation import cross_validation_splits

splits = None
if RUN_LIGHTGBM_TRAINING:
    splits = cross_validation_splits(
        table, phase_targets,
        mode="classification", n_splits=N_SPLITS, random_state=42,
    )
else:
    print("LightGBM training is disabled pending a full modelling table.")

LightGBM training is disabled pending a full modelling table.


### 3. Train phase classifiers

**Caution.** The original 10,000-round setting is intentionally retained and is expensive across all phase targets.

**Extension.** Use early stopping on a separately designed validation protocol if the objective changes from paper reproduction to model development.

In [3]:
from lc_qspr.lightgbm_training import (
    LightGBMConfig, train_lightgbm_phases,
)

phase_result = None
if RUN_LIGHTGBM_TRAINING:
    phase_result = train_lightgbm_phases(
        table, feature_names, phase_targets, splits,
        config=LightGBMConfig(iterations=10000),
        output_directory=RESULT_DIRECTORY / "lightgbm_phase_models",
        show_progress=True,
        families=families,
    )
    phase_result.predictions.to_csv(
        RESULT_DIRECTORY / "LightGBM_phase_oof.csv", index=True
    )
    display(phase_result.summary)

### 4. Train temperature regressors

**Caution.** Each target drops missing training labels independently, so report the observed sample count for every temperature.

**Extension.** Target-specific descriptors or monotonic constraints can be explored without changing the phase-classification workflow.

In [4]:
from lc_qspr.lightgbm_training import train_lightgbm_temperatures

temperature_result = None
if RUN_LIGHTGBM_TRAINING:
    temperature_result = train_lightgbm_temperatures(
        table, feature_names, temperature_targets, splits,
        config=LightGBMConfig(iterations=50000),
        output_directory=RESULT_DIRECTORY / "lightgbm_temperature_models",
        show_progress=True,
        families=families,
    )
    temperature_result.predictions.to_csv(
        RESULT_DIRECTORY / "LightGBM_temperature_oof.csv", index=True
    )
    display(temperature_result.summary)

### 5. Plot learning curves and inspect importance

**Caution.** A flat validation curve does not prove generalisation; compare folds and retain molecule-level out-of-fold errors.

**Extension.** Median fold importance is less sensitive to one unusual split than importance from a single model.

In [5]:
from lc_qspr.plotting import plot_lightgbm_histories

if phase_result is not None:
    plot_lightgbm_histories(
        phase_result.evaluation_history,
        metric_names=["multi_logloss", "accuracy"],
    )
    first_target = phase_targets[0]
    display(phase_result.feature_importance[first_target].head(20))

### 6. Tune a selected target with Optuna

**Caution.** Repeated tuning against one random hold-out can overfit the hold-out; reserve a final test set or use nested cross-validation.

**Extension.** The same objective supports binary phase classification and temperature regression through the `mode` argument.

In [6]:
from lc_qspr.tuning import tune_lightgbm

study = None
if RUN_OPTUNA_TUNING:
    tuning_target = "Np"
    observed = table[tuning_target].notna()
    study = tune_lightgbm(
        table.loc[observed, feature_names],
        table.loc[observed, tuning_target],
        mode="regression", trials=100, iterations=5000,
    )
    print(study.best_value)
    print(study.best_params)
else:
    print("Optuna tuning is disabled; enable it independently of model fitting.")

Optuna tuning is disabled; enable it independently of model fitting.
